# Normalisation des variables numériques

## Objectif

Ce notebook a pour objectif de normaliser les variables numériques du jeu de données CICIDS2017, en s'appuyant sur les ensembles d'entraînement et de test générés lors de l'étape précédente (`03_preparation_apprentissage.ipynb`).

Les variables du jeu de données présentent des échelles très hétérogènes (par exemple, `Flow Duration` peut atteindre plusieurs dizaines de millions, tandis que des indicateurs binaires comme `SYN Flag Count` ne prennent que les valeurs 0 ou 1). Cette hétérogénéité peut biaiser l'apprentissage de certains modèles de Machine Learning sensibles à l'échelle des données (régression logistique, SVM, k-plus-proches-voisins, réseaux de neurones), qui accorderaient artificiellement plus de poids aux variables présentant les plus grandes valeurs numériques.

La normalisation vise donc à ramener l'ensemble des variables explicatives à une échelle comparable (moyenne nulle et écart-type unitaire), sans altérer les relations et la structure des données.

Ce notebook réalise les étapes suivantes :

1. **Chargement** des ensembles `X_train`, `X_test`, `y_train` et `y_test` générés précédemment.
2. **Normalisation** des variables explicatives à l'aide de la standardisation (\textit{StandardScaler}), ajustée uniquement sur l'ensemble d'entraînement afin d'éviter toute fuite de données vers l'ensemble de test.
3. **Vérification** de la normalisation obtenue.
4. **Sauvegarde** des données normalisées, prêtes à être utilisées pour l'entraînement des modèles de Machine Learning.

## 1) Chargement des ensembles train/test

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

X_train = pd.read_csv("./data/processed/X_train.csv")
X_test = pd.read_csv("./data/processed/X_test.csv")
y_train = pd.read_csv("./data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("./data/processed/y_test.csv").squeeze()

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

X_train : (2017889, 47)
X_test  : (504473, 47)
y_train : (2017889,)
y_test  : (504473,)


## 2) Normalisation des variables numériques

La standardisation (StandardScaler) est ajustée (fit) uniquement sur `X_train`, puis appliquée (transform) à la fois sur `X_train` et `X_test`, afin d'éviter toute fuite de données de l'ensemble de test vers l'entraînement.

In [2]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Normalisation terminée.")

Normalisation terminée.


## 3) Vérification de la normalisation

Après normalisation, la moyenne de chaque variable doit être proche de 0 et l'écart-type proche de 1.

In [3]:
print("Moyenne après normalisation (train) :", X_train_scaled.mean().round(4))
print("Écart-type après normalisation (train) :", X_train_scaled.std().round(4))

Moyenne après normalisation (train) : 0.0
Écart-type après normalisation (train) : 1.0


## 4) Sauvegarde des données normalisées

Les données normalisées sont sauvegardées au format CSV, ainsi que l'objet `scaler` (au format .pkl), afin de pouvoir appliquer la même transformation à de nouvelles données lors du déploiement du modèle.

In [5]:
import os
import joblib

os.makedirs("./data/processed", exist_ok=True)

pd.DataFrame(X_train_scaled, columns=X_train.columns).to_csv("./data/processed/X_train_scaled.csv", index=False)
pd.DataFrame(X_test_scaled, columns=X_test.columns).to_csv("./data/processed/X_test_scaled.csv", index=False)

joblib.dump(scaler, "./data/processed/scaler.pkl")

print("Données normalisées et scaler sauvegardés avec succès.")

Données normalisées et scaler sauvegardés avec succès.
